In [9]:
import subprocess
from pathlib import Path

GITHUB_USERNAME = "Priyasahaana"
GITHUB_EMAIL = "2026aj05309@wilp.bits-pilani.ac.in"

REPO_URL = (
    "https://github.com/Priyasahaana/"
    "2026AJ05309_ML_Assignment2_ShopSense_Online_Purchase_Intention.git"
)

REPO_DIR = Path("/content/2026AJ05309_ShopSense_Online_Purchase_Intention")

print("GitHub repository:", REPO_URL)
print("Local working directory:", REPO_DIR)

GitHub repository: https://github.com/Priyasahaana/2026AJ05309_ML_Assignment2_ShopSense_Online_Purchase_Intention.git
Local working directory: /content/2026AJ05309_ShopSense_Online_Purchase_Intention


In [10]:
from google.colab import userdata
GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise ValueError(
        "GITHUB_TOKEN not found. Check Colab Secrets and notebook access."
    )
subprocess.run(
    ["git", "config", "--global", "credential.helper", "store"],
    check=True
)
credential_text = (
    "protocol=https\n"
    "host=github.com\n"
    f"username={GITHUB_USERNAME}\n"
    f"password={GITHUB_TOKEN}\n\n"
)
subprocess.run(
    ["git", "credential", "approve"],
    input=credential_text,
    text=True,
    check=True
)
del GITHUB_TOKEN
print("GitHub authentication configured for this Colab runtime.")

GitHub authentication configured for this Colab runtime.


In [11]:
if REPO_DIR.exists():
    print("Repository folder already exists. Reusing it.")
else:
    subprocess.run(
        ["git", "clone", REPO_URL, str(REPO_DIR)],
        check=True
    )
    print("Repository cloned successfully.")
subprocess.run(
    ["git", "-C", str(REPO_DIR), "checkout", "-b", "main"],
    check=True
)
subprocess.run(
    ["git", "-C", str(REPO_DIR), "config", "user.name", GITHUB_USERNAME],
    check=True
)
subprocess.run(
    ["git", "-C", str(REPO_DIR), "config", "user.email", GITHUB_EMAIL],
    check=True
)
print("Main branch and commit identity configured.")

Repository folder already exists. Reusing it.
Main branch and commit identity configured.


In [12]:
subprocess.run(
    ["git", "-C", str(REPO_DIR), "status"],
    check=True
)
print("\nRemote repository:")
subprocess.run(
    ["git", "-C", str(REPO_DIR), "remote", "-v"],
    check=True
)


Remote repository:


CompletedProcess(args=['git', '-C', '/content/2026AJ05309_ShopSense_Online_Purchase_Intention', 'remote', '-v'], returncode=0)

## 1. Environment setup
Install the libraries used for data loading, analysis, modelling, evaluation, and saving model artifacts. We do not force a protobuf version: this notebook does not need TensorFlow, and Streamlit is installed separately for deployment.

In [13]:
%pip install -q ucimlrepo scikit-learn pandas numpy matplotlib seaborn joblib scipy

## 2. Imports, reproducibility, and project folders

In [14]:
from pathlib import Path
import json, sys, zipfile, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sklearn

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score, recall_score,
    f1_score, matthews_corrcoef, confusion_matrix,
    classification_report, ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
TEST_SIZE = 0.20
PROJECT_DIR = Path("ShopSense")
MODEL_DIR = PROJECT_DIR / "model"
ASSET_DIR = PROJECT_DIR / "assets"
for folder in [PROJECT_DIR, MODEL_DIR, ASSET_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__, "| NumPy:", np.__version__, "| Scikit-learn:", sklearn.__version__)
print("Project folder:", PROJECT_DIR.resolve())

Python: 3.13.18
Pandas: 2.2.3 | NumPy: 2.4.6 | Scikit-learn: 1.6.1
Project folder: /content/ShopSense


## 3. Load the UCI dataset
The preferred route uses `ucimlrepo` and UCI dataset ID **468**. If the environment cannot connect to UCI, use a local CSV named `online_shoppers_intention.csv`.

In [15]:
LOCAL_DATA_PATH = Path("online_shoppers_intention.csv")

if LOCAL_DATA_PATH.exists():
    raw_df = pd.read_csv(LOCAL_DATA_PATH)
    print("Loaded local CSV:", LOCAL_DATA_PATH.resolve())
else:
    try:
        from ucimlrepo import fetch_ucirepo
        uci_dataset = fetch_ucirepo(id=468)
        raw_df = pd.concat(
            [uci_dataset.data.features.copy(), uci_dataset.data.targets.copy()],
            axis=1
        )
        print("Downloaded UCI dataset ID 468.")
    except Exception as exc:
        raise RuntimeError(
            "UCI download failed. Download the Online Shoppers Purchasing Intention "
            "CSV, save it beside this notebook as 'online_shoppers_intention.csv', "
            f"and rerun this cell. Original error: {exc}"
        )

raw_df.columns = [str(c).strip() for c in raw_df.columns]
if "Revenue" not in raw_df.columns:
    raise ValueError(f"Expected target column 'Revenue'. Columns found: {raw_df.columns.tolist()}")

df = raw_df.copy()
if pd.api.types.is_bool_dtype(df["Revenue"]):
    df["Revenue"] = df["Revenue"].astype(int)
else:
    target_map = {True: 1, False: 0, "true": 1, "false": 0, "1": 1, "0": 0, 1: 1, 0: 0}
    converted = df["Revenue"].map(
        lambda v: target_map.get(v, target_map.get(str(v).strip().lower(), np.nan))
    )
    if converted.isna().any():
        raise ValueError(f"Unrecognized Revenue values: {df['Revenue'].dropna().unique()[:10]}")
    df["Revenue"] = converted.astype(int)

print("Dataset shape:", df.shape)
display(df.head())

Downloaded UCI dataset ID 468.
Dataset shape: (12330, 18)


,Administrative,Administrative_Duration,Informational,Informational_Duration,ProductRelated,ProductRelated_Duration,BounceRates,ExitRates,PageValues,SpecialDay,Month,OperatingSystems,Browser,Region,TrafficType,VisitorType,Weekend,Revenue
0,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,1,1,1,1,Returning_Visitor,False,0
1,0,0.0,0,0.0,2,64.000000,0.00,0.10,0.0,0.0,Feb,2,2,1,2,Returning_Visitor,False,0
2,0,0.0,0,0.0,1,0.000000,0.20,0.20,0.0,0.0,Feb,4,1,9,3,Returning_Visitor,False,0
3,0,0.0,0,0.0,2,2.666667,0.05,0.14,0.0,0.0,Feb,3,2,2,4,Returning_Visitor,False,0
4,0,0.0,0,0.0,10,627.500000,0.02,0.05,0.0,0.0,Feb,3,3,1,4,Returning_Visitor,True,0


## 4. Verify dataset size and assignment minimums

In [16]:
target_column = "Revenue"
feature_columns = [c for c in df.columns if c != target_column]
print("Instances:", len(df))
print("Input features:", len(feature_columns))
print("Target classes:", sorted(df[target_column].unique().tolist()))
assert len(df) >= 500, "Dataset does not meet the 500-instance minimum."
assert len(feature_columns) >= 12, "Dataset does not meet the 12-feature minimum."
assert set(df[target_column].unique()).issubset({0, 1})
print("Assignment minimums satisfied.")

Instances: 12330
Input features: 17
Target classes: [0, 1]
Assignment minimums satisfied.


In [19]:
from google.colab import drive
drive.mount('/content/drive')
import subprocess
NOTEBOOK_NAME = "2026AJ05309_MACHINE_LEARNING_AIMLZG565_Assignment2.ipynb"
result = subprocess.run(
    ["find", "/content/drive/MyDrive", "-type", "f", "-name", NOTEBOOK_NAME],
    capture_output=True,
    text=True,
    check=True
)
NOTEBOOK_PATH = Path(result.stdout.strip().splitlines()[0]) if result.stdout.strip() else None
if NOTEBOOK_PATH is None:
    print("Notebook not found. Check its name and Google Drive location.")
else:
    print("Notebook found:", NOTEBOOK_PATH)

Mounted at /content/drive
Notebook found: /content/drive/MyDrive/Colab Notebooks/2026AJ05309_MACHINE_LEARNING_AIMLZG565_Assignment2.ipynb
